In [ ]:
# !pip install "git+https://github.com/AllenNeuralDynamics/aind-dynamic-foraging-database.git"

In [ ]:
import time

import pandas as pd
import duckdb

from aind_dynamic_foraging_database import SESSION_DB, TRIAL_DB, EVENT_DB

# SESSION_DB / TRIAL_DB / EVENT_DB point at the public production cache on S3.
# Reassign them to a local directory to query a local build instead.

# Reusable snippets - union_by_name handles schema differences across NWB reader paths.
READ_TRIALS = f"read_parquet('{TRIAL_DB}/**/*.parquet', hive_partitioning=true, union_by_name=true)"
READ_EVENTS = f"read_parquet('{EVENT_DB}/**/*.parquet', hive_partitioning=true, union_by_name=true)"

In [ ]:
# Helpers (imported alongside the paths from the setup cell above).
from aind_dynamic_foraging_database import (
    select_sessions, fetch_trials, fetch_events, read_trials,
)

# Common pattern: provide your own list of subjects and fetch just their final / graduated-stage
# sessions (`current_stage_actual`). `subjects=` keeps the read scoped to those mice, so it stays
# fast — the speed-up shrinks as the selection widens to many subjects.
my_subjects = ["801997", "801995", "792292", "739195"]
grad = select_sessions(
    "current_stage_actual IN ('GRADUATED', 'STAGE_FINAL')",
    subjects=my_subjects,
    columns=["current_stage_actual", "curriculum_name"],
)
print(f"{len(grad)} final/graduated sessions across {grad['subject_id'].nunique()} subjects")
grad_trials = fetch_trials(grad, columns=["animal_response", "earned_reward"])
print(f"{len(grad_trials):,} trials")
grad_trials.head()

In [ ]:
# select all sessions, all columns

sessions = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{SESSION_DB}')
    ORDER BY subject_id, session_date
""").df()

sessions

In [ ]:
MIN_SESSIONS = 10   # configurable

session_counts = sessions.groupby('subject_id')['session_date'].transform('size')
sessions_enough = sessions[session_counts >= MIN_SESSIONS]
sessions_enough


In [ ]:
for col in sessions.columns:
    print(col)

In [ ]:
len(sessions)

In [ ]:
sessions['subject_id'].nunique()

In [ ]:
sessions_enough['subject_id'].nunique()

In [ ]:
tbl = (sessions.groupby('curriculum_name', dropna=False)
              .agg(n_subjects=('subject_id', 'nunique'),
                   n_sessions=('subject_id', 'size'))
              .T)
tbl['total'] = tbl.sum(axis=1)
tbl


In [ ]:
# 1. count each task across a subject's sessions, pick the most common (ignoring None/NaN/"None")
def top_task(s):
    s = s.dropna()
    s = s[~s.astype(str).str.strip().str.lower().isin({'none', 'nan', ''})]
    return s.mode().iloc[0] if not s.empty else pd.NA

subject_curriculum = (sessions.groupby('subject_id')['task']
                              .agg(top_task)
                              .rename('subject_curriculum'))

# 2. attach the assigned major curriculum back to every session
s2 = sessions.merge(subject_curriculum, on='subject_id', how='left')

# 3. table: subjects & sessions per assigned curriculum
tbl2 = (s2.groupby('subject_curriculum', dropna=False)
          .agg(n_subjects=('subject_id', 'nunique'),
               n_sessions=('subject_id', 'size'))
          .T)
tbl2['total'] = tbl2.sum(axis=1)
tbl2


In [ ]:
tbl3 = (sessions_enough.groupby('curriculum_name', dropna=False)
              .agg(n_subjects=('subject_id', 'nunique'),
                   n_sessions=('subject_id', 'size'))
              .T)
tbl3['total'] = tbl3.sum(axis=1)
tbl3


In [ ]:
# 1. count each task across a subject's sessions, pick the most common (ignoring None/NaN/"None")
subject_curriculum_enough = (sessions_enough.groupby('subject_id')['task']
                              .agg(top_task)
                              .rename('subject_curriculum'))

# 2. attach the assigned major curriculum back to every session
s4 = sessions_enough.merge(subject_curriculum_enough, on='subject_id', how='left')

# 3. table: subjects & sessions per assigned curriculum
tbl4 = (s4.groupby('subject_curriculum', dropna=False)
          .agg(n_subjects=('subject_id', 'nunique'),
               n_sessions=('subject_id', 'size'))
          .T)
tbl4['total'] = tbl4.sum(axis=1)
tbl4
